# GRAND: Graph Neural Diffusion

Self-contained, modern-PyTorch implementation of:

> Chamberlain, Rowbottom, Gorinova, Webb, Rossi, Bronstein. **GRAND: Graph Neural Diffusion**. ICML 2021.  
> Paper: https://arxiv.org/abs/2106.10934 ·

### All variants implemented

| Variant | Block | Description |
|---|---|---|
| `grand_l` | linear | Original GRAND-l: fixed RW-normalized adjacency |
| `grand_nl` | attention | Original GRAND-nl: softmax multi-head attention |
| `grand_nls` | signed | Signed-tanh attention — allows anti-smoothing |
| `grand_nls_gated` | gated | Learned per-edge mix of softmax + signed-tanh |
| `grand_l_pn`, `grand_nl_pn`, `grand_nls_pn` | per-node α,β | Each node has its own diffusivity |
| `grand_nl_pf`, `grand_nls_pf` | per-feature α,β | Each hidden channel has its own diffusivity |
| **`grand_struct_deg`** | structural | Softmax + β·(negated normalized degree difference) |
| **`grand_struct_cn`** | structural | Softmax + β·(normalized common-neighbors count) |
| **`grand_struct_jac`** | structural | Softmax + β·(Jaccard similarity of neighborhoods) |

### Datasets supported
- **Homophilic**: Cora, Citeseer, Pubmed, ogbn-arxiv
- **Heterophilic** (geom-gcn splits): Texas, Cornell, Wisconsin, Chameleon, Squirrel


## 1. Setup

In [ ]:
# Uncomment first time you run
# !pip install torch torch_geometric torchdiffeq ogb

In [2]:
import time, copy
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch_geometric.datasets import Planetoid, WebKB, WikipediaNetwork
from torch_geometric.utils import (add_remaining_self_loops, scatter, softmax,
                                   to_undirected, degree)
from torch_geometric.utils.num_nodes import maybe_num_nodes
from torch_geometric.transforms import GDC, TwoHop
from torch_geometric.nn import GCNConv, GATConv
from torchdiffeq import odeint, odeint_adjoint

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
print('Torch:', torch.__version__)
import torch_geometric; print('PyG  :', torch_geometric.__version__)

Device: cuda
Torch: 2.10.0+cu128
PyG  : 2.7.0


## 2. Graph normalization (RW with self-loops)

In [3]:
def get_rw_adj(edge_index, edge_weight=None, norm_dim=1, fill_value=1.0,
               num_nodes=None, dtype=None):
    num_nodes = maybe_num_nodes(edge_index, num_nodes)
    if edge_weight is None:
        edge_weight = torch.ones(edge_index.size(1), dtype=dtype,
                                 device=edge_index.device)
    if fill_value != 0:
        edge_index, edge_weight = add_remaining_self_loops(
            edge_index, edge_weight, fill_value, num_nodes)
    row, col = edge_index[0], edge_index[1]
    idx = row if norm_dim == 0 else col
    deg_v = scatter(edge_weight, idx, dim=0, dim_size=num_nodes, reduce='sum')
    deg_inv = deg_v.pow(-1)
    deg_inv.masked_fill_(deg_inv == float('inf'), 0.)
    if norm_dim == 0:
        edge_weight = deg_inv[idx] * edge_weight
    else:
        edge_weight = edge_weight * deg_inv[idx]
    return edge_index, edge_weight

## 3. Generic α, β parameters

`alpha_mode` controls how diffusivity is parameterized:
- `'scalar'` — single learnable α and β
- `'per_node'` — one α and β per node (with small init noise to break symmetry)
- `'per_feature'` — one α and β per hidden channel

In [4]:
def make_alpha_beta_params(opt, num_nodes, hidden_dim):
    init_a = 0.5 if opt.get('no_alpha_sigmoid', False) else 0.0
    mode = opt.get('alpha_mode', 'scalar')
    noise_scale = opt.get('per_node_init_noise', 0.05)
    if mode == 'scalar':
        return torch.tensor(init_a), torch.tensor(0.0)
    elif mode == 'per_node':
        a = torch.full((num_nodes,), init_a) + noise_scale * torch.randn(num_nodes)
        b = noise_scale * torch.randn(num_nodes)
        return a, b
    elif mode == 'per_feature':
        a = torch.full((hidden_dim,), init_a) + noise_scale * torch.randn(hidden_dim)
        b = noise_scale * torch.randn(hidden_dim)
        return a, b
    else:
        raise ValueError(f'Unknown alpha_mode: {mode}')


def apply_alpha(alpha_train, x, opt):
    a = torch.sigmoid(alpha_train) if not opt['no_alpha_sigmoid'] else alpha_train
    mode = opt.get('alpha_mode', 'scalar')
    if mode in ('scalar', 'per_feature'):
        return a * x
    else:  # per_node
        return a.unsqueeze(1) * x


def apply_beta(beta_train, x0, opt):
    mode = opt.get('alpha_mode', 'scalar')
    if mode in ('scalar', 'per_feature'):
        return beta_train * x0
    else:
        return beta_train.unsqueeze(1) * x0

## 4. Structural similarity features

For the structural-attention variants, we precompute a per-edge similarity score $s(i,j)$ from the input graph topology. This is a fixed property of the graph — computed once, used at every ODE step.

Three choices:
- **`degree_diff`**: $s(i,j) = -|deg(i) - deg(j)| / d_{max}$ (negated and normalized so similar degrees → larger $s$)
- **`common_neighbors`**: $s(i,j) = |N(i) \cap N(j)| / d_{max}$
- **`jaccard`**: $s(i,j) = |N(i) \cap N(j)| / |N(i) \cup N(j)|$ — most interpretable, in $[0, 1]$

In [5]:
def compute_structural_features(edge_index, num_nodes, kind='jaccard'):
    """Return per-edge structural similarity s(i,j), shape [num_edges]."""
    src, dst = edge_index[0], edge_index[1]
    deg_v = degree(src, num_nodes=num_nodes)

    if kind == 'degree_diff':
        max_d = deg_v.max().item() + 1
        d_diff = (deg_v[src] - deg_v[dst]).abs() / max_d
        return -d_diff   # negate so similar degrees give larger value

    # For CN and Jaccard we need set intersection per edge.
    # Vectorized via dense adjacency — fine for graphs up to ~30k nodes.
    n = num_nodes
    # adj = torch.sparse_coo_tensor(
    #     edge_index, torch.ones(edge_index.size(1)), size=(n, n)
    # ).coalesce().to_dense().bool()
    adj = torch.sparse_coo_tensor(
        edge_index,
        torch.ones(edge_index.size(1), device=edge_index.device),
        size=(n, n),
    ).coalesce().to_dense().bool()
    src_nbr = adj[src]
    dst_nbr = adj[dst]
    intersection = (src_nbr & dst_nbr).sum(dim=1).float()

    if kind == 'common_neighbors':
        max_d = deg_v.max().item() + 1
        return intersection / max_d
    if kind == 'jaccard':
        union = (src_nbr | dst_nbr).sum(dim=1).float()
        return intersection / (union + 1e-12)
    raise ValueError(f'Unknown structural kind: {kind}')

## 5. GRAND-l ODE function (linear)

In [6]:
class LaplacianODEFunc(nn.Module):
    def __init__(self, hidden_dim, opt, num_nodes_hint=None):
        super().__init__()
        self.opt = opt
        self.hidden_dim = hidden_dim
        self.edge_index = self.edge_weight = self.num_nodes = self.x0 = None
        self.nfe = 0
        if opt.get('alpha_mode', 'scalar') == 'per_node' and num_nodes_hint is None:
            self.alpha_train = self.beta_train = None
        else:
            a, b = make_alpha_beta_params(opt, num_nodes_hint or 1, hidden_dim)
            self.alpha_train = nn.Parameter(a)
            self.beta_train = nn.Parameter(b)

    def init_per_node_params(self, num_nodes):
        if self.alpha_train is None:
            init_a = 0.5 if self.opt['no_alpha_sigmoid'] else 0.0
            noise_scale = self.opt.get('per_node_init_noise', 0.05)
            a_init = torch.full((num_nodes,), init_a) + noise_scale * torch.randn(num_nodes)
            b_init = noise_scale * torch.randn(num_nodes)
            self.alpha_train = nn.Parameter(a_init)
            self.beta_train = nn.Parameter(b_init)

    def sparse_mul(self, x):
        adj = torch.sparse_coo_tensor(
            self.edge_index, self.edge_weight,
            size=(self.num_nodes, self.num_nodes)).coalesce()
        return torch.sparse.mm(adj, x)

    def forward(self, t, x):
        if self.nfe > self.opt['max_nfe']:
            raise RuntimeError(f"Max NFE ({self.opt['max_nfe']}) exceeded")
        self.nfe += 1
        ax = self.sparse_mul(x)
        f = apply_alpha(self.alpha_train, ax - x, self.opt)
        if self.opt['add_source']:
            f = f + apply_beta(self.beta_train, self.x0, self.opt)
        return f

## 6. Attention layer

Supports five attention types:
- **`scaled_dot`** (`grand_nl`): standard softmax attention
- **`signed_tanh`** (`grand_nls*`): tanh + ℓ₁ normalization, allows negative weights
- **`gated`** (`grand_nls_gated`): per-edge mix of softmax and signed-tanh
- **`structural`** (`grand_struct_*`): softmax with structural bias added to logit

In [7]:
def signed_tanh_norm(prods, index, num_nodes=None):
    N = maybe_num_nodes(index, num_nodes)
    t = torch.tanh(prods)
    abs_t = t.abs()
    if t.dim() == 1:
        denom = scatter(abs_t, index, dim=0, dim_size=N, reduce='sum')[index]
    else:
        denom = torch.stack([
            scatter(abs_t[:, h], index, dim=0, dim_size=N, reduce='sum')[index]
            for h in range(abs_t.size(1))
        ], dim=1)
    return t / (denom + 1e-16)


class SpGraphTransAttentionLayer(nn.Module):
    def __init__(self, in_features, out_features, opt, edge_weights=None):
        super().__init__()
        self.opt = opt
        self.h = int(opt['heads'])
        self.attention_dim = opt.get('attention_dim', out_features)
        assert self.attention_dim % self.h == 0
        self.d_k = self.attention_dim // self.h
        self.edge_weights = edge_weights
        if opt['attention_type'] == 'exp_kernel':
            self.output_var = nn.Parameter(torch.ones(1))
            self.lengthscale = nn.Parameter(torch.ones(1))
        self.Q = nn.Linear(in_features, self.attention_dim)
        self.K = nn.Linear(in_features, self.attention_dim)
        self.V = nn.Linear(in_features, self.attention_dim)
        self.Wout = nn.Linear(self.d_k, in_features)
        for m in (self.Q, self.K, self.V, self.Wout):
            nn.init.constant_(m.weight, 1e-5)

        if opt['attention_type'] == 'gated':
            self.gate_lin = nn.Linear(2 * in_features, 1)
            nn.init.normal_(self.gate_lin.weight, mean=0.0, std=0.01)
            nn.init.zeros_(self.gate_lin.bias)

        # Structural-attention learnable beta — mixes content + structural logits
        if opt['attention_type'] == 'structural':
            self.struct_beta = nn.Parameter(torch.tensor(1.0))

        # Per-edge structural features (set externally by ODE block)
        self.struct_features = None

    def forward(self, x, edge, x0=None):
        q = self.Q(x).view(-1, self.h, self.d_k).transpose(1, 2)
        k = self.K(x).view(-1, self.h, self.d_k).transpose(1, 2)
        v = self.V(x).view(-1, self.h, self.d_k).transpose(1, 2)
        src = q[edge[0]]; dst = k[edge[1]]

        at = self.opt['attention_type']
        if at in ('scaled_dot', 'signed_tanh', 'gated', 'structural'):
            prods = (src * dst).sum(dim=1) / np.sqrt(self.d_k)
        elif at == 'cosine_sim':
            prods = nn.CosineSimilarity(dim=1, eps=1e-5)(src, dst)
        elif at == 'exp_kernel':
            prods = self.output_var ** 2 * torch.exp(
                -((src - dst) ** 2).sum(dim=1) / (2 * self.lengthscale ** 2))
        elif at == 'pearson':
            src_c = src - src.mean(dim=1, keepdim=True)
            dst_c = dst - dst.mean(dim=1, keepdim=True)
            prods = nn.CosineSimilarity(dim=1, eps=1e-5)(src_c, dst_c)
        else:
            raise ValueError(f'Unknown attention_type: {at}')

        if self.opt['reweight_attention'] and self.edge_weights is not None:
            prods = prods * self.edge_weights.unsqueeze(1)

        # Add structural bias to the logit before softmax
        if at == 'structural':
            assert self.struct_features is not None, \
                'struct_features must be set on attention layer before forward'
            prods = prods + self.struct_beta * self.struct_features.unsqueeze(1)

        idx = edge[self.opt['attention_norm_idx']]
        if at == 'signed_tanh':
            attention = signed_tanh_norm(prods, idx)
        elif at == 'gated':
            soft = softmax(prods, idx)
            sign = signed_tanh_norm(prods, idx)
            gate_features = x0 if x0 is not None else x
            gate_input = torch.cat([gate_features[edge[0]], gate_features[edge[1]]], dim=1)
            g = torch.sigmoid(self.gate_lin(gate_input))
            attention = g * soft + (1 - g) * sign
        else:
            attention = softmax(prods, idx)
        return attention, v


class ODEFuncTransformerAtt(nn.Module):
    def __init__(self, in_features, out_features, opt, num_nodes_hint=None):
        super().__init__()
        self.opt = opt
        self.in_features = in_features
        self.out_features = out_features
        self.edge_index = self.edge_weight = self.num_nodes = self.x0 = None
        self.nfe = 0
        if opt.get('alpha_mode', 'scalar') == 'per_node' and num_nodes_hint is None:
            self.alpha_train = self.beta_train = None
        else:
            a, b = make_alpha_beta_params(opt, num_nodes_hint or 1, in_features)
            self.alpha_train = nn.Parameter(a)
            self.beta_train = nn.Parameter(b)
        self.multihead_att_layer = None

    def init_per_node_params(self, num_nodes):
        if self.alpha_train is None:
            init_a = 0.5 if self.opt['no_alpha_sigmoid'] else 0.0
            noise_scale = self.opt.get('per_node_init_noise', 0.05)
            a_init = torch.full((num_nodes,), init_a) + noise_scale * torch.randn(num_nodes)
            b_init = noise_scale * torch.randn(num_nodes)
            self.alpha_train = nn.Parameter(a_init)
            self.beta_train = nn.Parameter(b_init)

    def multiply_attention(self, x, attention):
        mean_att = attention.mean(dim=1)
        adj = torch.sparse_coo_tensor(
            self.edge_index, mean_att,
            size=(self.num_nodes, self.num_nodes)).coalesce()
        return torch.sparse.mm(adj, x)

    def forward(self, t, x):
        if self.nfe > self.opt['max_nfe']:
            raise RuntimeError(f"Max NFE ({self.opt['max_nfe']}) exceeded")
        self.nfe += 1
        attention, _ = self.multihead_att_layer(x, self.edge_index, x0=self.x0)
        ax = self.multiply_attention(x, attention)
        f = apply_alpha(self.alpha_train, ax - x, self.opt)
        if self.opt['add_source']:
            f = f + apply_beta(self.beta_train, self.x0, self.opt)
        return f

## 7. ODE block + GRAND model

The block sets up edges, normalization, structural features (if needed), and per-node parameter init.

In [8]:
class ConstantODEBlock(nn.Module):
    def __init__(self, odefunc, opt, data, device):
        super().__init__()
        self.opt = opt
        self.device = device
        self.odefunc = odefunc

        if getattr(data, 'edge_attr', None) is not None:
            ei, ew = data.edge_index, data.edge_attr
            if opt['self_loop_weight'] > 0:
                ei, ew = add_remaining_self_loops(
                    ei, ew, fill_value=opt['self_loop_weight'],
                    num_nodes=data.num_nodes)
        else:
            ei, ew = get_rw_adj(
                data.edge_index, edge_weight=None, norm_dim=1,
                fill_value=opt['self_loop_weight'],
                num_nodes=data.num_nodes, dtype=data.x.dtype)

        self.odefunc.edge_index = ei.to(device)
        self.odefunc.edge_weight = ew.to(device)
        self.odefunc.num_nodes = data.num_nodes

        if hasattr(self.odefunc, 'init_per_node_params'):
            self.odefunc.init_per_node_params(data.num_nodes)
            self.odefunc.alpha_train.data = self.odefunc.alpha_train.data.to(device)
            self.odefunc.beta_train.data = self.odefunc.beta_train.data.to(device)

        if isinstance(self.odefunc, ODEFuncTransformerAtt):
            self.odefunc.multihead_att_layer = SpGraphTransAttentionLayer(
                self.odefunc.in_features, self.odefunc.out_features,
                opt, edge_weights=self.odefunc.edge_weight).to(device)
            # Compute structural features on the (post self-loop) edge_index
            if opt['attention_type'] == 'structural':
                kind = opt['structural_kind']
                s = compute_structural_features(
                    self.odefunc.edge_index, data.num_nodes, kind=kind).to(device)
                self.odefunc.multihead_att_layer.struct_features = s

        self.register_buffer('t', torch.tensor([0.0, opt['time']]))
        self.atol = opt['tol_scale'] * 1e-7
        self.rtol = opt['tol_scale'] * 1e-9

    def set_x0(self, x0):
        self.odefunc.x0 = x0.clone().detach()

    _FIXED = {'rk4', 'euler', 'midpoint', 'explicit_adams', 'implicit_adams'}

    def forward(self, x, t_vec=None):
        t = self.t.type_as(x) if t_vec is None else t_vec.type_as(x)
        integrator = odeint_adjoint if self.opt['adjoint'] else odeint
        kwargs = dict(method=self.opt['method'], atol=self.atol, rtol=self.rtol)
        if self.opt['method'] in self._FIXED:
            kwargs['options'] = {'step_size': self.opt['step_size']}
        if self.opt['adjoint']:
            kwargs['adjoint_method'] = self.opt['adjoint_method']
            kwargs['adjoint_atol'] = self.opt['tol_scale_adjoint'] * 1e-7
            kwargs['adjoint_rtol'] = self.opt['tol_scale_adjoint'] * 1e-9
            if self.opt['adjoint_method'] in self._FIXED:
                kwargs['adjoint_options'] = {'step_size': self.opt['adjoint_step_size']}
        z = integrator(self.odefunc, x, t, **kwargs)
        return z if t_vec is not None else z[-1]


# Maps variant name -> (block, attention_type, alpha_mode, no_sigmoid, add_source, structural_kind)
VARIANT_SPEC = {
    'grand_l':           ('constant',  None,           'scalar',      False, True,  None),
    'grand_l_pn':        ('constant',  None,           'per_node',    False, True,  None),
    'grand_l_pf':        ('constant',  None,           'per_feature', False, True,  None),
    'grand_nl':          ('attention', 'scaled_dot',   'scalar',      False, True,  None),
    'grand_nl_pn':       ('attention', 'scaled_dot',   'per_node',    False, True,  None),
    'grand_nl_pf':       ('attention', 'scaled_dot',   'per_feature', False, True,  None),
    'grand_nls':         ('attention', 'signed_tanh',  'scalar',      True,  False, None),
    'grand_nls_pn':      ('attention', 'signed_tanh',  'per_node',    True,  False, None),
    'grand_nls_pf':      ('attention', 'signed_tanh',  'per_feature', True,  False, None),
    'grand_nls_gated':   ('attention', 'gated',        'scalar',      True,  False, None),
    'grand_struct_deg':  ('attention', 'structural',   'scalar',      False, True,  'degree_diff'),
    'grand_struct_cn':   ('attention', 'structural',   'scalar',      False, True,  'common_neighbors'),
    'grand_struct_jac':  ('attention', 'structural',   'scalar',      False, True,  'jaccard'),
}
ALL_VARIANTS = list(VARIANT_SPEC.keys())
ALL_VARIANTS2 = ['grand_struct_jac','grand_struct_cn','grand_struct_jac']


class GRAND(nn.Module):
    def __init__(self, opt, dataset, device):
        super().__init__()
        self.opt = opt
        hd = opt['hidden_dim']
        self.num_classes = dataset.num_classes
        self.enc = nn.Linear(dataset.num_features, hd)
        self.dec = nn.Linear(hd, self.num_classes)
        n_hint = dataset[0].num_nodes if opt.get('alpha_mode') == 'per_node' else None
        block = opt['block']
        if block == 'constant':
            func = LaplacianODEFunc(hd, opt, num_nodes_hint=n_hint)
        elif block == 'attention':
            func = ODEFuncTransformerAtt(hd, hd, opt, num_nodes_hint=n_hint)
        else:
            raise ValueError(f"Unknown block: {block}")
        self.odeblock = ConstantODEBlock(func, opt, dataset[0], device).to(device)

    def forward(self, x, return_trajectory_at=None):
        x = F.dropout(x, self.opt['input_dropout'], training=self.training)
        x = self.enc(x)
        self.odeblock.set_x0(x)
        if return_trajectory_at is not None:
            return self.odeblock(x, t_vec=return_trajectory_at)
        z = self.odeblock(x)
        z = F.relu(z)
        z = F.dropout(z, self.opt['dropout'], training=self.training)
        return self.dec(z)

    def reset_nfe(self): self.odeblock.odefunc.nfe = 0
    @property
    def nfe(self): return self.odeblock.odefunc.nfe

## 8. Graph rewiring

In [9]:
def apply_gdc(data, opt):
    if opt['gdc_method'] == 'ppr':
        diff_args = dict(method='ppr', alpha=opt['ppr_alpha'])
    else:
        diff_args = dict(method='heat', t=opt['heat_timapply_gdce'])
    if opt['gdc_sparsification'] == 'topk':
        sparse_args = dict(method='topk', k=opt['gdc_k'], dim=0)
        diff_args['eps'] = opt['gdc_threshold']
    else:
        sparse_args = dict(method='threshold', eps=opt['gdc_threshold'])
        diff_args['eps'] = opt['gdc_threshold']
    slw = opt['self_loop_weight']
    gdc = GDC(self_loop_weight=slw if slw > 0 else None,
              normalization_in='sym', normalization_out='col',
              diffusion_kwargs=diff_args, sparsification_kwargs=sparse_args,
              exact=opt.get('exact', True))
    return gdc(data)

def apply_two_hop(data):
    return TwoHop()(data)

def rewire(data, opt):
    r = opt.get('rewiring', 'none')
    if r == 'gdc':
        before = data.num_edges
        data = apply_gdc(data, opt)
        print(f"GDC ({opt['gdc_method']}, k={opt['gdc_k']}): edges {before} -> {data.num_edges}")
    elif r == 'two_hop':
        before = data.num_edges
        data = apply_two_hop(data)
        print(f"Two-hop: edges {before} -> {data.num_edges}")
    elif r in (None, 'none'):
        pass
    else:
        raise ValueError(f"Unknown rewiring: {r}")
    return data

## 9. Configuration

In [10]:
BASE = dict(
    hidden_dim=64, input_dropout=0.5, dropout=0.4,
    optimizer='adam', lr=0.01, weight_decay=5e-4,
    epochs=100, patience=100,
    time=4.0, self_loop_weight=1.0, add_source=True, no_alpha_sigmoid=False,
    method='rk4', step_size=0.5, adjoint=False,
    adjoint_method='rk4', adjoint_step_size=1.0,
    tol_scale=1.0, tol_scale_adjoint=1.0, max_nfe=2000,
    block='constant', alpha_mode='scalar',
    heads=4, attention_dim=64, attention_type='scaled_dot',
    attention_norm_idx=1, reweight_attention=False, leaky_relu_slope=0.2,
    rewiring='none',
    gdc_method='ppr', ppr_alpha=0.05, heat_time=3.0,
    gdc_sparsification='topk', gdc_k=64, gdc_threshold=0.01, exact=True,
    structural_kind='jaccard',
    per_node_init_noise=0.05,
)

DATASET_OPT = {
    'Cora':       dict(hidden_dim=64, input_dropout=0.5, dropout=0.1,
                       time=4.0, lr=0.01, weight_decay=5e-3, epochs=100,
                       heads=4, attention_dim=64),
    'Citeseer':   dict(hidden_dim=64, input_dropout=0.5, dropout=0.5,
                       time=4.0, lr=0.01, weight_decay=0.05, epochs=200,
                       heads=4, attention_dim=64),
    'Pubmed':     dict(hidden_dim=128, input_dropout=0.5, dropout=0.1,
                       time=4.0, lr=0.01, weight_decay=2e-3, epochs=200,
                       adjoint=True, heads=4, attention_dim=128),
    'ogbn-arxiv': dict(hidden_dim=128, input_dropout=0.1, dropout=0.1,
                       time=3.0, lr=0.005, weight_decay=0.0, epochs=100,
                       adjoint=True, heads=2, attention_dim=128, max_nfe=500),
    'Texas':      dict(hidden_dim=64, input_dropout=0.3, dropout=0.5,
                       time=3.0, lr=0.01, weight_decay=5e-4, epochs=200,
                       heads=4, attention_dim=64),
    'Cornell':    dict(hidden_dim=64, input_dropout=0.3, dropout=0.5,
                       time=3.0, lr=0.01, weight_decay=5e-4, epochs=200,
                       heads=4, attention_dim=64),
    'Wisconsin':  dict(hidden_dim=64, input_dropout=0.3, dropout=0.5,
                       time=3.0, lr=0.01, weight_decay=5e-4, epochs=200,
                       heads=4, attention_dim=64),
    'Chameleon':  dict(hidden_dim=64, input_dropout=0.3, dropout=0.3,
                       time=3.0, lr=0.005, weight_decay=5e-4, epochs=200,
                       heads=4, attention_dim=64),
    'Squirrel':   dict(hidden_dim=64, input_dropout=0.3, dropout=0.3,
                       time=3.0, lr=0.005, weight_decay=5e-4, epochs=200,
                       heads=4, attention_dim=64),
}

REWIRING_PRESETS = {
    'none':     dict(rewiring='none'),
    'gdc':      dict(rewiring='gdc', gdc_method='ppr', ppr_alpha=0.05,
                     gdc_sparsification='topk', gdc_k=64, gdc_threshold=0.01, exact=True),
    'gdc_heat': dict(rewiring='gdc', gdc_method='heat', heat_time=3.0,
                     gdc_sparsification='topk', gdc_k=64, gdc_threshold=0.01, exact=True),
    'two_hop':  dict(rewiring='two_hop'),
}


def make_opt(dataset, variant='grand_l', rewiring='none', **overrides):
    if variant not in VARIANT_SPEC:
        raise ValueError(f"Unknown variant: {variant}. Use one of {ALL_VARIANTS}")
    if dataset not in DATASET_OPT:
        raise ValueError(f"Unknown dataset: {dataset}")

    opt = copy.deepcopy(BASE)
    opt.update(DATASET_OPT[dataset])
    block, attn, alpha_mode, no_sig, add_src, struct = VARIANT_SPEC[variant]
    opt['block'] = block
    if attn is not None:
        opt['attention_type'] = attn
    opt['alpha_mode'] = alpha_mode
    opt['no_alpha_sigmoid'] = no_sig
    opt['add_source'] = add_src
    if struct is not None:
        opt['structural_kind'] = struct
    opt.update(REWIRING_PRESETS[rewiring])
    opt.update(overrides)
    opt['dataset'] = dataset
    opt['variant'] = variant
    return opt

## 10. Data loading

In [11]:
HETEROPHILIC = {'Texas', 'Cornell', 'Wisconsin', 'Chameleon', 'Squirrel'}


class _WrappedDataset:
    def __init__(self, data, num_features, num_classes):
        self._data = data
        self.num_features = num_features
        self.num_classes = num_classes
    def __getitem__(self, i): return self._data


def load_dataset(name, root='./data', split_idx=0, verbose=True):
    path = f'{root}/{name}'
    if name in ('Cora', 'Citeseer', 'Pubmed'):
        dataset = Planetoid(path, name)
        num_feats, num_classes = dataset.num_features, dataset.num_classes
        data = dataset[0]
    elif name == 'ogbn-arxiv':
        from ogb.nodeproppred import PygNodePropPredDataset
        dataset = PygNodePropPredDataset(name=name, root=path)
        data = dataset[0]
        data.edge_index = to_undirected(data.edge_index)
        sp = dataset.get_idx_split()
        n = data.num_nodes
        def mk(idx):
            m = torch.zeros(n, dtype=torch.bool); m[idx] = True; return m
        data.train_mask = mk(sp['train']); data.val_mask = mk(sp['valid'])
        data.test_mask = mk(sp['test'])
        data.y = data.y.squeeze()
        num_feats, num_classes = dataset.num_features, dataset.num_classes
    elif name in ('Texas', 'Cornell', 'Wisconsin'):
        dataset = WebKB(path, name)
        data = dataset[0]
        data.train_mask = data.train_mask[:, split_idx]
        data.val_mask   = data.val_mask[:, split_idx]
        data.test_mask  = data.test_mask[:, split_idx]
        num_feats, num_classes = dataset.num_features, dataset.num_classes
    elif name in ('Chameleon', 'Squirrel'):
        dataset = WikipediaNetwork(path, name.lower(), geom_gcn_preprocess=True)
        data = dataset[0]
        data.train_mask = data.train_mask[:, split_idx]
        data.val_mask   = data.val_mask[:, split_idx]
        data.test_mask  = data.test_mask[:, split_idx]
        num_feats, num_classes = dataset.num_features, dataset.num_classes
    else:
        raise ValueError(f'Unknown dataset: {name}')

    if verbose:
        print(f"{name}: {data.num_nodes} nodes, {data.num_edges} edges, "
              f"{num_feats} feats, {num_classes} classes"
              + (f", split={split_idx}" if name in HETEROPHILIC else ""))
    return _WrappedDataset(data, num_feats, num_classes)

## 11. Training utilities

In [12]:
def get_optimizer(name, params, lr, weight_decay):
    return {'adam': torch.optim.Adam, 'adamax': torch.optim.Adamax,
            'rmsprop': torch.optim.RMSprop,
            'sgd': torch.optim.SGD}[name](params, lr=lr, weight_decay=weight_decay)


def train_step(model, optim, data):
    model.train(); model.reset_nfe(); optim.zero_grad()
    out = model(data.x)
    loss = F.cross_entropy(out[data.train_mask], data.y[data.train_mask])
    loss.backward(); optim.step()
    return loss.item(), model.nfe


@torch.no_grad()
def evaluate(model, data):
    model.eval()
    logits = model(data.x)
    accs = []
    for mask in (data.train_mask, data.val_mask, data.test_mask):
        pred = logits[mask].argmax(-1)
        accs.append((pred == data.y[mask]).float().mean().item())
    return accs


def train_grand_once(opt, raw_dataset, device, verbose=False):
    d = rewire(raw_dataset[0].clone(), opt) if opt.get('rewiring','none') != 'none' \
        else raw_dataset[0].clone()
    ds = _WrappedDataset(d, raw_dataset.num_features, raw_dataset.num_classes)
    d = d.to(device)
    m = GRAND(opt, ds, device).to(device)
    o = get_optimizer(opt['optimizer'], m.parameters(), opt['lr'], opt['weight_decay'])
    best_val = best_test = 0.0
    for ep in range(1, opt['epochs'] + 1):
        train_step(m, o, d)
        _, v, t = evaluate(m, d)
        if v > best_val:
            best_val, best_test = v, t
            if verbose:
                print(f'  ep {ep:03d}  val {v:.3f}  test {t:.3f}')
    return best_val, best_test, m

In [13]:
import time

print_training_time=False

if print_training_time:
    print(f'\n=== Training time per epoch (Cora, mean of 10 epochs) ===')
    raw = load_dataset('Cora', verbose=False)
    for variant in ALL_VARIANTS:
        torch.manual_seed(0)
        opt = make_opt('Cora', variant=variant, rewiring='none', epochs=10)
        d = raw[0].clone().to(device)
        m = GRAND(opt, raw, device).to(device)
        o = get_optimizer(opt['optimizer'], m.parameters(),
                          opt['lr'], opt['weight_decay'])
        # Warm up
        for _ in range(2):
            train_step(m, o, d)
        if device.type == 'cuda':
            torch.cuda.synchronize()
        t0 = time.time()
        nfes = []
        for _ in range(10):
            _, nfe = train_step(m, o, d)
            nfes.append(nfe)
        if device.type == 'cuda':
            torch.cuda.synchronize()
        elapsed_ms = (time.time() - t0) * 100  # per epoch in ms
        print(f'  {variant:20s}  {elapsed_ms:6.0f} ms/epoch  '
              f'NFE={np.mean(nfes):.0f}')

## 12. Baselines: GCN, GAT, MLP

In [14]:
class GCN(nn.Module):
    def __init__(self, in_ch, hidden, out_ch, dropout=0.5):
        super().__init__()
        self.conv1 = GCNConv(in_ch, hidden, cached=False)
        self.conv2 = GCNConv(hidden, out_ch, cached=False)
        self.dropout = dropout
    def forward(self, x, edge_index):
        x = F.relu(self.conv1(x, edge_index))
        x = F.dropout(x, self.dropout, training=self.training)
        return self.conv2(x, edge_index)

class GAT(nn.Module):
    def __init__(self, in_ch, hidden, out_ch, heads=8, dropout=0.6):
        super().__init__()
        self.conv1 = GATConv(in_ch, hidden, heads=heads, dropout=dropout)
        self.conv2 = GATConv(hidden*heads, out_ch, heads=1, concat=False, dropout=dropout)
        self.dropout = dropout
    def forward(self, x, edge_index):
        x = F.dropout(x, self.dropout, training=self.training)
        x = F.elu(self.conv1(x, edge_index))
        x = F.dropout(x, self.dropout, training=self.training)
        return self.conv2(x, edge_index)

class MLP(nn.Module):
    def __init__(self, in_ch, hidden, out_ch, dropout=0.5):
        super().__init__()
        self.fc1 = nn.Linear(in_ch, hidden)
        self.fc2 = nn.Linear(hidden, out_ch)
        self.dropout = dropout
    def forward(self, x, edge_index=None):
        x = F.relu(self.fc1(x))
        x = F.dropout(x, self.dropout, training=self.training)
        return self.fc2(x)

def train_baseline(model_cls, data, num_features, num_classes, device,
                   hidden=64, lr=0.01, wd=5e-4, epochs=200, **mkwargs):
    model = model_cls(num_features, hidden, num_classes, **mkwargs).to(device)
    optim = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
    d = data.to(device)
    best_val = best_test = 0.0
    for ep in range(1, epochs + 1):
        model.train(); optim.zero_grad()
        out = model(d.x, d.edge_index)
        loss = F.cross_entropy(out[d.train_mask], d.y[d.train_mask])
        loss.backward(); optim.step()
        model.eval()
        with torch.no_grad():
            logits = model(d.x, d.edge_index)
            v = (logits[d.val_mask].argmax(-1) == d.y[d.val_mask]).float().mean().item()
            t = (logits[d.test_mask].argmax(-1) == d.y[d.test_mask]).float().mean().item()
        if v > best_val:
            best_val, best_test = v, t
    return best_val, best_test

## 13. Multi-seed evaluation

In [15]:
def multi_seed_eval(dataset_name, variant, rewiring='none', n_seeds=5,
                    device=device, overrides=None, verbose=True):
    overrides = overrides or {}
    test_accs = []
    is_hetero = dataset_name in HETEROPHILIC
    for seed in range(n_seeds):
        torch.manual_seed(seed); np.random.seed(seed)
        split = seed % 10 if is_hetero else 0
        try:
            raw = load_dataset(dataset_name, split_idx=split,
                               verbose=(verbose and seed == 0))
            opt = make_opt(dataset_name, variant=variant, rewiring=rewiring, **overrides)
            _, test, _ = train_grand_once(opt, raw, device)
            test_accs.append(test)
            if verbose:
                print(f'  seed {seed} (split {split}): test {test:.4f}')
        except Exception as e:
            if verbose:
                print(f'  seed {seed}: FAILED ({type(e).__name__}: {e})')
    if not test_accs:
        return None, None, []
    arr = np.array(test_accs)
    return arr.mean(), arr.std(), test_accs


def multi_seed_baseline(dataset_name, model_cls, n_seeds=5, device=device,
                        hidden=64, lr=0.01, wd=5e-4, epochs=200,
                        verbose=True, **mkwargs):
    test_accs = []
    is_hetero = dataset_name in HETEROPHILIC
    for seed in range(n_seeds):
        torch.manual_seed(seed); np.random.seed(seed)
        split = seed % 10 if is_hetero else 0
        raw = load_dataset(dataset_name, split_idx=split,
                           verbose=(verbose and seed == 0))
        _, t = train_baseline(model_cls, raw[0], raw.num_features, raw.num_classes,
                              device, hidden=hidden, lr=lr, wd=wd, epochs=epochs,
                              **mkwargs)
        test_accs.append(t)
        if verbose:
            print(f'  seed {seed} (split {split}): test {t:.4f}')
    arr = np.array(test_accs)
    return arr.mean(), arr.std(), test_accs

## 14. Single-run quick start

Choose any variant (including the new structural ones), pick a dataset, train.

In [16]:
DATASET  = 'Cora'
VARIANT  = 'grand_struct_jac' 
REWIRING = 'none'
SEED     = 0

torch.manual_seed(SEED); np.random.seed(SEED)
opt = make_opt(DATASET, variant=VARIANT, rewiring=REWIRING)
raw = load_dataset(DATASET, split_idx=SEED % 10 if DATASET in HETEROPHILIC else 0)
print(f"\nRunning {VARIANT} on {DATASET} (rewiring={REWIRING})")

bv, bt, model = train_grand_once(opt, raw, device, verbose=True)
print(f'\nBest val {bv:.4f}  test @ best val {bt:.4f}')

# For structural variants, inspect the learned beta
if 'struct' in VARIANT:
    beta = model.odeblock.odefunc.multihead_att_layer.struct_beta.item()
    print(f'Learned structural beta: {beta:.4f}')

Processing...
Done!


Cora: 2708 nodes, 10556 edges, 1433 feats, 7 classes

Running grand_struct_jac on Cora (rewiring=none)
  ep 001  val 0.236  test 0.250
  ep 002  val 0.572  test 0.588
  ep 003  val 0.658  test 0.664
  ep 004  val 0.696  test 0.707
  ep 005  val 0.720  test 0.732
  ep 006  val 0.742  test 0.759
  ep 007  val 0.750  test 0.773
  ep 008  val 0.760  test 0.786
  ep 009  val 0.780  test 0.786

Best val 0.7800  test @ best val 0.7860
Learned structural beta: 0.6112


In [17]:
RUN_FULL_SWEEP = False

SWEEP_DATASETS = ['Cora', 'Citeseer', 'Pubmed', 'Texas', 'Cornell', 'Wisconsin']
SWEEP_VARIANTS = ALL_VARIANTS
SWEEP_BASELINES = [('GCN', GCN), ('GAT', GAT), ('MLP', MLP)]
SWEEP_BASELINES = []
N_SEEDS = 10


def fmt(mu, sd):
    if mu is None:
        return '   n/a    '
    return f'{mu*100:5.1f}±{sd*100:4.1f}'


if RUN_FULL_SWEEP:
    results = {}
    n_runs = (len(SWEEP_VARIANTS) + len(SWEEP_BASELINES)) * len(SWEEP_DATASETS) * N_SEEDS
    print(f'Total runs: ~{n_runs}\n')

    for ds in SWEEP_DATASETS:
        print(f'\n=== {ds} ===')
        for v in SWEEP_VARIANTS:
            mu, sd, _ = multi_seed_eval(ds, v, 'none', n_seeds=N_SEEDS, verbose=True)
            results[(ds, v)] = (mu, sd)
            print(f'  {v:18s}  {fmt(mu, sd)}')
        for name, cls in SWEEP_BASELINES:
            mu, sd, _ = multi_seed_baseline(ds, cls, n_seeds=N_SEEDS, verbose=True)
            results[(ds, name)] = (mu, sd)
            print(f'  {name:18s}  {fmt(mu, sd)}')

    cols = SWEEP_VARIANTS + [n for n, _ in SWEEP_BASELINES]
    col_w = max(len(c) for c in cols) + 2
    print('\n' + '=' * (12 + col_w * len(cols)))
    print(f'{"Dataset":<12}' + ''.join(f'{c:>{col_w}s}' for c in cols))
    print('-' * (12 + col_w * len(cols)))
    for ds in SWEEP_DATASETS:
        row = f'{ds:<12}'
        for c in cols:
            mu, sd = results.get((ds, c), (None, None))
            row += f'{fmt(mu, sd):>{col_w}s}'
        print(row)
    print('=' * (12 + col_w * len(cols)))
else:
    print('RUN_FULL_SWEEP disabled. Set RUN_FULL_SWEEP=True to run.')

RUN_FULL_SWEEP disabled. Set RUN_FULL_SWEEP=True to run.


## 15. Multi-seed accuracy table — all 13 variants × all datasets

This is the main quantitative comparison. Set `RUN_FULL_SWEEP=True` to run.

## 16. Inspect learned structural beta

After training, what value did each structural variant's learnable beta converge to? If $|\beta|$ is large, structural information dominates the attention. If $\beta \to 0$, the model is ignoring structural information — equivalent to plain `grand_nl`.

In [18]:
INSPECT_STRUCT_BETA = False

if INSPECT_STRUCT_BETA:
    inspect_datasets = ['Cora', 'Citeseer', 'Texas', 'Cornell', 'Wisconsin']
    inspect_variants = ['grand_struct_deg', 'grand_struct_cn', 'grand_struct_jac']
    print(f'{"dataset":<12} {"variant":<22} {"learned beta":>14}')
    print('-' * 50)
    for ds in inspect_datasets:
        for v in inspect_variants:
            torch.manual_seed(0); np.random.seed(0)
            raw = load_dataset(ds, split_idx=0, verbose=False)
            opt = make_opt(ds, variant=v, rewiring='none')
            _, _, m = train_grand_once(opt, raw, device)
            beta = m.odeblock.odefunc.multihead_att_layer.struct_beta.item()
            print(f'{ds:<12} {v:<22} {beta:>14.4f}')
else:
    print('INSPECT_STRUCT_BETA disabled.')

INSPECT_STRUCT_BETA disabled.


In [ ]:
# === Depth experiment across multiple datasets ===
import matplotlib.pyplot as plt

GRAND_TIMES     = [1, 2, 4, 8, 16, 32, 64]
BASELINE_LAYERS = [2, 4, 8, 16, 32, 64]
DEPTH_DATASETS  = ['Wisconsin', 'Texas']
DEPTH_VARIANTS  = ['grand_l', 'grand_nl', 'grand_nls']
DEPTH_SEEDS     = 3


# Stackable GCN that takes a depth argument
class DeepGCN(nn.Module):
    def __init__(self, in_ch, hidden, out_ch, n_layers=2, dropout=0.5):
        super().__init__()
        self.dropout = dropout
        self.convs = nn.ModuleList()
        if n_layers == 1:
            self.convs.append(GCNConv(in_ch, out_ch, cached=False))
        else:
            self.convs.append(GCNConv(in_ch, hidden, cached=False))
            for _ in range(n_layers - 2):
                self.convs.append(GCNConv(hidden, hidden, cached=False))
            self.convs.append(GCNConv(hidden, out_ch, cached=False))

    def forward(self, x, edge_index):
        for i, conv in enumerate(self.convs):
            x = conv(x, edge_index)
            if i < len(self.convs) - 1:
                x = F.relu(x)
                x = F.dropout(x, self.dropout, training=self.training)
        return x


def train_grand_at_time(dataset_name, variant, T, n_seeds=3):
    accs = []
    is_hetero = dataset_name in HETEROPHILIC
    for seed in range(n_seeds):
        torch.manual_seed(seed); np.random.seed(seed)
        split = seed % 10 if is_hetero else 0
        try:
            raw = load_dataset(dataset_name, split_idx=split, verbose=False)
            opt = make_opt(dataset_name, variant=variant, rewiring='none',
                           time=float(T))
            _, t, _ = train_grand_once(opt, raw, device)
            accs.append(t)
        except Exception as e:
            print(f'    {variant}@T={T} seed={seed}: FAILED ({type(e).__name__})')
    return (np.mean(accs), np.std(accs)) if accs else (None, None)


def train_deepgcn(dataset_name, n_layers, n_seeds=3, dropout=0.5):
    accs = []
    is_hetero = dataset_name in HETEROPHILIC
    for seed in range(n_seeds):
        torch.manual_seed(seed); np.random.seed(seed)
        split = seed % 10 if is_hetero else 0
        raw = load_dataset(dataset_name, split_idx=split, verbose=False)
        d = raw[0].to(device)
        m = DeepGCN(raw.num_features, 64, raw.num_classes,
                    n_layers=n_layers, dropout=dropout).to(device)
        opt = torch.optim.Adam(m.parameters(), lr=0.01, weight_decay=5e-4)
        best_val = best_test = 0.0
        for _ in range(200):
            m.train(); opt.zero_grad()
            out = m(d.x, d.edge_index)
            loss = F.cross_entropy(out[d.train_mask], d.y[d.train_mask])
            loss.backward(); opt.step()
            m.eval()
            with torch.no_grad():
                logits = m(d.x, d.edge_index)
                v = (logits[d.val_mask].argmax(-1) == d.y[d.val_mask]).float().mean().item()
                t = (logits[d.test_mask].argmax(-1) == d.y[d.test_mask]).float().mean().item()
            if v > best_val:
                best_val, best_test = v, t
        accs.append(best_test)
    return np.mean(accs), np.std(accs)

    
print_plots=True
if print_plots:
    # === Run sweep ===
    all_results = {}   # {(dataset, variant): [(T, mu, sd), ...]}
    for ds in DEPTH_DATASETS:
        print(f'\n=== {ds} ===')
        for v in DEPTH_VARIANTS:
            print(f'  --- {v} ---')
            runs = []
            for T in GRAND_TIMES:
                mu, sd = train_grand_at_time(ds, v, T, DEPTH_SEEDS)
                runs.append((T, mu, sd))
                if mu is not None:
                    print(f'    T={T:3d}: {mu*100:5.1f} ± {sd*100:4.1f}')
            all_results[(ds, v)] = runs
    
        print(f'  --- DeepGCN ---')
        gcn_runs = []
        for L in BASELINE_LAYERS:
            mu, sd = train_deepgcn(ds, L, DEPTH_SEEDS)
            gcn_runs.append((L, mu, sd))
            print(f'    L={L:3d}: {mu*100:5.1f} ± {sd*100:4.1f}')
        all_results[(ds, 'deepgcn')] = gcn_runs
    # === Plot all datasets in a 2x2 grid (or 1xN if 4 or fewer) ===
    n = len(DEPTH_DATASETS)
    ncols = 2 if n > 2 else n
    nrows = (n + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(5.5 * ncols, 4 * nrows),
                             squeeze=False, sharey=False)
    
    colors = {'grand_l': 'C0', 'grand_nl': 'C1', 'grand_nls': 'C2', 'deepgcn': 'C3'}
    labels = {'grand_l':  'GRAND-L',
              'grand_nl': 'GRAND-NL',
              'grand_nls': 'GRAND-NLS (ours)',
              'deepgcn':  'DeepGCN (layers)'}
    linestyles = {'grand_l': '-', 'grand_nl': '-', 'grand_nls': '-', 'deepgcn': '--'}
    markers = {'grand_l': 'o', 'grand_nl': 'o', 'grand_nls': 'o', 'deepgcn': 's'}
    
    for i, ds in enumerate(DEPTH_DATASETS):
        ax = axes[i // ncols, i % ncols]
        for v in DEPTH_VARIANTS + ['deepgcn']:
            runs = all_results.get((ds, v), [])
            ts = [r[0] for r in runs if r[1] is not None]
            mus = [r[1] * 100 for r in runs if r[1] is not None]
            sds = [r[2] * 100 for r in runs if r[1] is not None]
            if ts:
                ax.errorbar(ts, mus, yerr=sds, marker=markers[v],
                            label=labels[v], color=colors[v],
                            linewidth=2, capsize=3, linestyle=linestyles[v])
        ax.set_xscale('log', base=2)
        ax.set_xlabel(r'integration time $T$ / layer count')
        ax.set_ylabel('test accuracy (%)')
        is_hetero = ds in HETEROPHILIC
        ax.set_title(f'{ds} ({"heterophilic" if is_hetero else "homophilic"})')
        ax.grid(alpha=0.3, which='both')
        if i == 0:
            ax.legend(loc='best', fontsize=9)
    
    # Hide unused subplots if any
    for j in range(n, nrows * ncols):
        axes[j // ncols, j % ncols].axis('off')
    
    plt.tight_layout()
    plt.savefig('depth_scaling_multi.pdf', bbox_inches='tight')
    plt.savefig('depth_scaling_multi.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    
    # === Print final summary table ===
    print('\n=== Final summary ===')
    header = f'{"dataset":<10s} {"method":<15s}' + ''.join(
        f'{T:>8s}' for T in [str(t) for t in GRAND_TIMES])
    print(header)
    print('-' * len(header))
    for ds in DEPTH_DATASETS:
        for v in DEPTH_VARIANTS + ['deepgcn']:
            runs = all_results.get((ds, v), [])
            cells = ''
            for r in runs:
                if r[1] is None:
                    cells += f'{"--":>8s}'
                else:
                    cells += f'{r[1]*100:>8.1f}'
            print(f'{ds:<10s} {v:<15s} {cells}')